# Module 6 Assignment: Give Your Persona Agent More Tools

In Module 5 you built a Persona Agent with one tool. This assignment makes the same jump Lab 5
made on top of Lab 4: your agent goes from one tool to three, and `chat()` has to learn to route
between them instead of assuming.

You'll reuse your own Module 5 code at almost every step, plus the routing pattern from Lab 5.
This assignment is about **extending** what you already built, not writing it from scratch.

**What you'll build:**
1. Your persona, notes, and first tool, carried over from Module 5
2. A second tool, backed by a free, no-key external API
3. A third tool of your own design
4. JSON descriptions for all three tools
5. An updated system prompt that tells the model when to use each tool
6. A `call_tool(name, arguments)` dispatcher, exactly like Lab 5's
7. A full agent loop that routes across all three tools

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you can find where Lab 5 made this same change to Lab 4, you can make it here too.

## 0. Install the packages (run once)

Same as Module 5, plus one new package: `requests`, for calling your new tool's API.

In [1]:
!uv pip install --upgrade openai pypdf gradio python-dotenv requests

Resolved 63 packages in 4.37s
 Downloaded openai
 Downloaded gradio
Prepared 8 packages in 44.09s
Uninstalled 8 packages in 17.17s
Installed 8 packages in 8.25s
 - filelock==4.0.8
 + filelock==4.0.12
 - gradio==6.29.0
 + gradio==6.29.1
 - gradio-client==2.7.1
 + gradio-client==2.7.2
 - huggingface-hub==1.33.0
 + huggingface-hub==2.1.1
 - markupsafe==3.0.3
 + markupsafe==3.0.4
 - openai==3.22.1
 + openai==3.24.0
 - pytz==2026.4
 + pytz==2026.5
 - tzdata==2026.4
 + tzdata==2026.5


## 1. Imports

**TODO:** Copy the imports from your Module 5 assignment, plus `requests` for calling an external
API. Add any other standard library module your second or third tool needs (for example `html`,
if the API you pick returns HTML-escaped text the way Open Trivia Database does).

In [2]:
from dotenv import load_dotenv
import os
import sys
import json

import requests
from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr

c:\Users\colby\Desktop\IT3025C Labs\Lab 5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Connect to Azure OpenAI via APIM

Identical to every previous lab and assignment. Copy this cell directly from Module 5.

In [3]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment - fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins 512e1c13


## 3. Recreate your persona, notes, and first tool from Module 5

Bring forward everything you already built: your `persona`, your `notes/notes.pdf` loading code,
your one tool function from Module 5, and that tool's JSON description. Nothing here needs to
change, this is your starting point.

**TODO:** Paste in, from your Module 5 assignment:
- Your `persona` variable
- Your `PdfReader("notes/notes.pdf")` loading code that builds `notes`
- Your Module 5 tool function (rename it if you like, but keep track of its name)
- That tool's JSON schema dictionary

**Hint:** copy your `notes/notes.pdf` file into this assignment's folder too, in a `notes`
subfolder, exactly like Lab 5 did with Lab 4's notes file.

In [5]:
persona = (
   "Finance Coach, a calm and practical study assistant who helps students review liquidity, working capital, cash management, and breakeven analysis. Finance Coach asks focused questions, explains formulas clearly, and helps students identify topics they should review again."
)

reader = PdfReader("Notes/module5_notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

def my_tool(value):
    print(f"Tool caleld to log review item: {value}")
    with open("review_items.txt", "a", encoding="utf-8") as f:
        f.write(value + "\n")
    return "Review item logged successfully."
print(persona)
print(notes[:300], "...")

Finance Coach, a calm and practical study assistant who helps students review liquidity, working capital, cash management, and breakeven analysis. Finance Coach asks focused questions, explains formulas clearly, and helps students identify topics they should review again.
Module 5 Study Notes: Liquidity, Working Capital, and Breakeven
This study sheet summarizes Module 06: Liquidity, Working Capital, and Breakeven. The
main ideas are how entrepreneurs measure short-term financial health, manage cash and
working capital, and calculate the revenue level needed for surv ...


In [6]:
my_tool_json = {
    "name": "my_tool",
    "description": (
        "Record a finance topic or quiz question the student should review after "
        "answering incorrectly, saying they do not know, or appearing unsure."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "value": {
                "type": "string",
                "description": "The finance topic or quiz question the student should review."
            }
        },
        "required": ["value"],
        "additionalProperties": False
    }
}

## 4. New tool: something backed by a free API

Design a **second** tool for your Persona Agent, this one calling a free, no-key external API,
the same way Lab 5's `fetch_bonus_question_tool` calls the Open Trivia Database.

**Ideas, pick whichever fits your topic, or find your own:**
- A trivia-friendly agent could call the Open Trivia Database for a bonus question
- A weather-aware topic could call [Open-Meteo](https://open-meteo.com), a free weather API that
  needs no key
- A language-learning agent could call the [Free Dictionary API](https://dictionaryapi.dev) to
  define a word, though it is sometimes slow or briefly unavailable, so handle that the same way
  the next tip describes
- Any other free, no-key API relevant to your topic works too, just check it does not require
  signup or a paid key

**TODO:** Write your tool function below. It should make one `requests.get(...)` call inside a
`try`/`except` that catches `requests.exceptions.RequestException`, so a slow or unavailable API
returns a short, friendly string instead of crashing your notebook, exactly like Lab 5's tools
do.

In [7]:
def api_tool(currency):
    print(f"Tool called to check the exchange rate for {currency}")

    currency = currency.upper()

    try:
        response = requests.get(
            f"https://api.frankfurter.dev/v2/rate/USD/{currency}",
            timeout=20
        )

        if response.status_code != 200:
            return "I could not find that exchange rate."

        data = response.json()

    except requests.exceptions.RequestException:
        return "The exchange rate website is not available right now."

    except ValueError:
        return "The exchange rate website returned invalid information."

    if "rate" not in data:
        return "I could not find an exchange rate for that currency."

    rate = data["rate"]

    return f"1 US dollar is about {rate} {currency}."

In [8]:
api_tool("EUR")  # quick manual test of your API-backed tool

Tool called to check the exchange rate for EUR


'1 US dollar is about 0.88965 EUR.'

## 5. New tool: one more tool of your own design

Design a **third** tool. It can be another local tool in the spirit of Module 5 (a running score
tracker, a bookmarking tool for topics that need review, or an idea of your own), or a second
external API call if you would rather. Either is fine, as long as it does something the first two
tools do not already cover.

**TODO:** Write your third tool function below.

In [9]:
def third_tool():
    print("Tool called to check review progress")

    try:
        with open("review_items.txt", "r", encoding="utf-8") as file:
            review_items = file.readlines()
    except FileNotFoundError:
        return "You do not have any review items yet."

    count = len(review_items)

    return f"You have {count} item(s) to review."

In [11]:
third_tool()  # quick manual test of your third tool

Tool called to check review progress


'You do not have any review items yet.'

## 6. Describe all three tools to the model with JSON

Same idea as Module 5 and Lab 5, just three schemas now. Remember an argument can be optional (an
empty `"required"` list), exactly like Lab 5's `fetch_bonus_question_tool`, if that fits your
tool.

In [12]:
api_tool_json = {
    "name": "api_tool",
    "description": "Use this tool when the student asks for an exchange rate.",
    "parameters": {
        "type": "object",
        "properties": {
            "currency": {
                "type": "string",
                "description": "The currency code, such as EUR, GBP, or JPY."
            }
        },
        "required": ["currency"],
        "additionalProperties": False
    }
}

third_tool_json = {
    "name": "third_tool",
    "description": "Use this tool when the student asks how many review items they have.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": my_tool_json},
    {"type": "function", "function": api_tool_json},
    {"type": "function", "function": third_tool_json}
]

## 7. Update your system prompt for three tools

**TODO:** Build your `system_prompt` f-string the same way you did in Module 5, still using
`persona` and `notes`, but now with three rules telling the model when to reach for each tool,
the same way Lab 5's system prompt added two new sentences to Lab 4's.

In [13]:
system_prompt = f"""
# Your role

You are {persona}

# Context

Use these notes to quiz the student:

{notes}

# Rules

Ask only one question at a time.

Only ask quiz questions based on the notes.

Wait for the student to answer before asking another question.

Tell the student if their answer is correct or incorrect.

Give a short explanation after they answer.

If the student gets a question wrong or says "I don't know",
use my_tool to save the question for review.

If the student asks for an exchange rate,
use api_tool to find the exchange rate.

If the student asks how many review items they have,
use third_tool to check their progress.

Keep your answers calm, simple, and encouraging.
"""

display(Markdown(system_prompt))



# Your role

You are Finance Coach, a calm and practical study assistant who helps students review liquidity, working capital, cash management, and breakeven analysis. Finance Coach asks focused questions, explains formulas clearly, and helps students identify topics they should review again.

# Context

Use these notes to quiz the student:

Module 5 Study Notes: Liquidity, Working Capital, and Breakeven
This study sheet summarizes Module 06: Liquidity, Working Capital, and Breakeven. The
main ideas are how entrepreneurs measure short-term financial health, manage cash and
working capital, and calculate the revenue level needed for survival or operating
breakeven.
Financial Ratios and Liquidity
Financial ratio analysis uses relationships among financial statement accounts to judge
a company’s condition and performance. Ratios can help entrepreneurs, lenders, and
investors see whether the business is becoming stronger or weaker over time.
Liquidity describes how easily assets can be converted into cash without losing value.
Cash is the most liquid asset. Receivables and marketable securities are also relatively
liquid. Fixed assets such as equipment, real estate, and specialized property are
usually less liquid because they take more time and effort to sell.
Accounting liquidity measures whether a company can meet short-term obligations with
liquid assets. A venture can be profitable but still face liquidity problems if cash is
tied up in inventory, receivables, or fixed assets.
Common Liquidity Measures
Current ratio = Current Assets / Current Liabilities. It shows whether current assets
could cover short-term liabilities. A current ratio above 1.0 usually means current
assets exceed current liabilities, but it does not guarantee all assets can quickly
become cash.
Quick ratio = (Current Assets - Inventory) / Current Liabilities. This is stricter than
the current ratio because it removes inventory, which may not be easy to sell quickly.
Cash ratio = Cash and Marketable Securities / Current Liabilities. This is the most
conservative liquidity ratio because it only counts funds that are immediately
available.
Net working capital = Current Assets - Current Liabilities. NWC is a dollar measure of
the company’s short-term cushion. NWC-to-total-assets ratio compares the working capital
buffer with the size of the company.
Cash Burn and Liquidity Events
Cash burn measures how quickly a venture uses cash. Monthly cash burn helps estimate how
long the company can continue operating with its available cash. If cash burn grows
faster than cash build, the company may become less liquid even while sales are
increasing.
A liquidity event allows investors or owners to cash out some or all ownership. Examples
include selling the company, selling to a strategic partner, going public, or selling an
interest to a private equity investor.
Working Capital Management
Page 1 of 3Working capital management is the management of short-term investment and financing. Its
goals are to maintain enough cash flow for operations and use resources productively.
Liquidity management means generating cash when and where it is needed. Drags on
liquidity delay cash collection, such as slow customer payments or obsolete inventory.
Pulls on liquidity cause cash to leave too soon, such as early payments to suppliers or
reduced access to credit.
Primary sources of liquidity include cash balances, short-term financing, trade credit,
bank loans, and cash flow management. Secondary sources include renegotiating debt,
selling assets, or bankruptcy reorganization.
Cash Conversion Cycle and Cash Management
The cash conversion cycle is the length of time it takes for inventory investment to
generate cash, considering that some inventory may be purchased on credit. Longer
operating and cash conversion cycles increase the need for liquidity.
Managers forecast short-term cash flows, monitor cash balances, and maintain target cash
balances to avoid negative cash positions. Cash inflows include customer receipts,
investment maturities, debt proceeds, and tax refunds. Cash outflows include payroll,
payables, investments, debt repayment, interest, dividends, and taxes.
Short-term investments can temporarily store excess funds. Managers consider liquidity,
maturity, credit risk, yield, and collateral requirements when choosing short-term
investments.
Receivables, Inventory, and Payables
Accounts receivable management focuses on efficient records, accurate collections,
communication with credit managers, and reporting. Companies may use credit terms such
as net days, cash before delivery, cash on delivery, bill-to-bill, or monthly billing.
An aging schedule shows how long receivables have been outstanding. Number of days
receivable should be compared with the company’s credit terms and competitors.
Inventory management seeks enough inventory to meet demand without holding more than
necessary. Motives for inventory include transaction needs, precaution against
stockouts, and speculation about price or availability. Common approaches include
economic order quantity, just-in-time systems, and materials/resource planning.
Accounts payable are a spontaneous form of credit from suppliers. Managers compare
number of days payable with credit terms and consider trade credit costs, vendor
relationships, alternative financing, and disbursement float.
Breakeven Terms
Variable expenses change directly with revenues, such as cost of goods sold. Fixed
expenses stay relatively constant over a relevant revenue range, such as administrative
costs, marketing costs, or some financing costs.
Variable Cost Revenue Ratio, or VCRR, equals Variable Costs / Revenues. Contribution
margin equals 1 - VCRR. A higher contribution margin means more of each revenue dollar
Page 2 of 3is available to cover fixed costs.
Cash flow breakeven means cash in equals cash out for a period. EBDAT means earnings
before depreciation, amortization, and taxes. EBDAT breakeven is the survival revenue
needed to cover cash operating expenses and fixed financing costs.
EBDAT and NOPAT Breakeven
EBDAT = Revenues - Variable Costs - Cash Fixed Costs. At EBDAT breakeven, EBDAT equals
zero, so Revenues = Variable Costs + Cash Fixed Costs.
Survival Revenues = Cash Fixed Costs / (1 - VCRR). For example, if cash fixed costs are
$400,000 and VCRR is 65%, then contribution margin is 35% and survival revenues are
about $1,143,000.
NOPAT means net operating profit after taxes. NOPAT breakeven focuses on operations
rather than financing. NOPAT breakeven revenues = Total Operating Fixed Costs / (1 -
VCRR). Total operating fixed costs include cash operating fixed costs plus noncash
operating costs such as depreciation, but exclude interest expense.
EBDAT breakeven is especially important for short-term survival because failure to cover
cash costs signals financial distress. Failure to cover noncash depreciation is not
necessarily an immediate cash crisis.
Key Takeaways
- Liquidity is about having enough usable cash or near-cash assets to meet short-term
obligations.
- The current ratio, quick ratio, cash ratio, and net working capital help diagnose
liquidity strength, but each has limitations.
- Working capital management focuses on cash, receivables, inventory, payables, short-term
financing, and the cash conversion cycle.
- Breakeven analysis shows how much revenue a venture needs to survive or cover operating
costs.
- Contribution margin is central to breakeven. If the contribution margin rises, the
revenue needed to break even falls.
Page 3 of 3

# Rules

Ask only one question at a time.

Only ask quiz questions based on the notes.

Wait for the student to answer before asking another question.

Tell the student if their answer is correct or incorrect.

Give a short explanation after they answer.

If the student gets a question wrong or says "I don't know",
use my_tool to save the question for review.

If the student asks for an exchange rate,
use api_tool to find the exchange rate.

If the student asks how many review items they have,
use third_tool to check their progress.

Keep your answers calm, simple, and encouraging.


## 8. The routing problem: which tool did the model actually call?

With three tools, `chat()` can no longer assume. It has to check `tool_call.function.name` and
call the matching Python function.

**TODO:** Copy the `call_tool(name, arguments)` dispatcher pattern from Lab 5, Step 9, but route
across your own three tool names instead.

In [14]:
def call_tool(name, arguments):

    if name == "my_tool":
        value = arguments.get("value")
        return my_tool(value)

    elif name == "api_tool":
        currency = arguments.get("currency")
        return api_tool(currency)

    elif name == "third_tool":
        return third_tool()

    else:
        return "Unknown tool"

## 9. Update `chat()` to route across all three tools

**TODO:** Copy the finished agent loop from your Module 5 assignment. The `while` loop and the
`for tool_call in message_obj.tool_calls` loop do not change at all, the only new line is a call
to `call_tool(...)` in place of your old single hardcoded tool call.

In [15]:
def chat(message, history):

    messages = [
        {"role": "system", "content": system_prompt}
    ] + history + [
        {"role": "user", "content": message}
    ]

    response = azure_client.chat.completions.create(
        model=DEPLOYMENT,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":

        message_obj = response.choices[0].message
        messages.append(message_obj)

        for tool_call in message_obj.tool_calls:

            name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            result = call_tool(name, arguments)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

        response = azure_client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

## 10. Try your multi-tool Persona Agent

Launch the chat interface and try to trigger all three tools in one session: something that
should hit your original Module 5 tool, something that should hit your new API-backed tool, and
something that should hit your third tool. Watch the terminal or output to see which tool fires
each time. Interrupt the cell when you're done.

In [16]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool caleld to log review item: What is net working capital, and what formula is used to calculate it?
Tool called to check the exchange rate for EUR
Tool called to check review progress


## Reflection questions

Answer these in a new Markdown cell below.

1. What are your three tools, and what does each one do? Why did you choose the second and third
   tools you designed?
2. Compare this assignment to Module 5. What was exactly the same as going from one tool to more
   than one? What, if anything, surprised you?
3. Walk through what `call_tool` actually does the first time your agent calls your new
   API-backed tool. What would break if you deleted the `elif` branch for that tool?
4. Open Lab 5's system prompt and compare it to yours. Both add rules for two new tools, using
   only plain English sentences and no code changes to the agent loop. Why does that work, in
   terms of what a system prompt actually is?

1. My first tool is `my_tool`. It saves a question or finance topic to a text file when I get an answer wrong or say that I do not know the answer.

My second tool is `api_tool`. It uses a free online API to find the exchange rate from US dollars to another currency. I chose this tool because exchange rates relate to finance and it gave me practice getting information from an outside source.

My third tool is `third_tool`. It reads my review file and tells me how many items I need to review. I chose this tool because it works with my first tool and helps the student check their progress.

2. A lot of this assignment was the same as Module 5. I used the same Finance Coach persona, PDF notes, system prompt, Azure OpenAI connection, and original review tool. I also used JSON to describe the tools to the AI.

The main difference was going from one tool to three tools. I needed a way to check which tool the AI wanted to use. The `call_tool` function does this by looking at the tool's name and running the correct Python function.

I was surprised that most of the agent loop stayed the same. Adding more tools did not require rebuilding the entire program. I mainly needed to add the new tool functions, their JSON descriptions, and the routing function.

3. When the agent decides it needs an exchange rate, it creates a tool call with the name `api_tool`. It also provides an argument containing the currency code.The `call_tool` function receives the tool name and its arguments. It checks the name using the `if` and `elif` statements. When it finds the `api_tool` branch, it gets the currency from the arguments and passes it to the `api_tool` Python function.

The API tool then contacts the exchange-rate website and returns the result. That result is added to the messages so the agent can explain it to the user.

If I deleted the `elif` branch for `api_tool`, the dispatcher would not know how to run that tool. It would reach the `else` branch and return `"Unknown tool"` instead of returning an exchange rate.

4. The system prompt tells the AI how it should behave and when it should use each tool. Lab 5's prompt explains when to use its bonus-question and progress tools. My prompt explains when to use the exchange-rate and review-progress tools.
This works because the system prompt gives instructions to the AI in plain English. The AI reads those instructions along with the descriptions in the `tools` list and decides which tool is appropriate for the user's request.

The agent loop does not need separate instructions for each situation. It is designed to accept any tool call, read the tool's name and arguments, run it through `call_tool`, and send the result back to the AI. This allows the behavior to change by adding new prompt rules and tool descriptions without changing the main agent loop.

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit, it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, for example `Module6Assignment_JaneDoe.ipynb`, and
  submit it.

You've now made the same jump Lab 5 made: one tool to several, with a real dispatcher and a loop
that already scaled to handle it.